In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.044 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 264.38it/s, Materializing param=model.norm.weight]                              


unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [2]:
import torch

new_tokens = [
    # --- Structural & Logic Tags ---
    "SALES_AGENT","Core Essentials", "30$", "5GB", "4G network connectivity", "Standard local voice calling", "Standard SMS messaging",
    "Urban Connect", "60$", "40GB", "5G network connectivity", "Unlimited local voice calls", "Mobile hotspot support", 
    "Apex Unlimited", "90$", "Unlimited Data", "Priority access to high-speed data", "International roaming", "Enterprise Elite",
    "110$", "Service-level agreement support", "Secure VPN access", "Student Basic", "25$", "10GB", "Unlimited access to selected social apps", 
    "Senior Basic", "35$", "Priority customer support", "THE_END", "Sorry, I can only help with mobile plans and services of amAIz Telecom.",
    "I'm a virtual assistant.", "How can I help you to select our plans?"
]

# Clean duplicates
new_tokens = list(set(new_tokens))

def initialize_new_tokens(model, tokenizer, new_tokens):
    # Get the embedding layer
    embeddings = model.get_input_embeddings()
    # Get the LM head
    lm_head = model.get_output_embeddings()
    
    for token in new_tokens:
        token_id = tokenizer.convert_tokens_to_ids(token)
        # Simple heuristic: average the embeddings of the characters in the token
        # This gives the model a much better "starting point" than random noise
        sub_tokens = tokenizer.tokenize(token[1:] if token.startswith(' ') else token)
        sub_token_ids = tokenizer.convert_tokens_to_ids(sub_tokens)
        
        if len(sub_token_ids) > 0:
            with torch.no_grad():
                avg_emb = embeddings.weight[sub_token_ids].mean(dim=0)
                embeddings.weight[token_id] = avg_emb
                
                avg_lm = lm_head.weight[sub_token_ids].mean(dim=0)
                lm_head.weight[token_id] = avg_lm

# Call this after resizing but BEFORE training
tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))
# model.resize_output_embeddings(len(tokenizer))
initialize_new_tokens(model, tokenizer, new_tokens)

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    modules_to_save = ["embed_tokens", "lm_head"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [4]:
######### data set for SFT

import json
from datasets import Dataset
import random

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

import json
import random
from datasets import Dataset

def prepare_data_set(file_name, tokenizer, max_seq_length):
    sft_data = []

    # 1. Load data
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                sft_data.append(json.loads(line))
    
    random.seed(42) 
    random.shuffle(sft_data)
    
    dataset = Dataset.from_dict({
        "messages": [item["messages"] for item in sft_data]
    })

    # 2. Format into ChatML strings
    def formatting_prompts_func(examples):
        output_texts = []
        for i in range(len(examples['messages'])):
            # add_generation_prompt=False is correct here because we want 
            # the full conversation (including the assistant's answer)
            text = tokenizer.apply_chat_template(
                examples['messages'][i], 
                tokenize=False, 
                add_generation_prompt=False
            )
            # Ensure the sequence ends with EOS
            if not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            output_texts.append(text)
        return { "text" : output_texts }

    # Map the formatting
    dataset = dataset.map(formatting_prompts_func, batched=True)

    # 3. Handle Tokenization
    # Note: We do NOT remove the 'text' column yet. 
    # SFTTrainer needs 'text' to handle completion-only masking.
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_seq_length,
            padding=False,
        )

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        # Keep 'text' column for Phase 2 collator!
        # Only remove 'messages' as it's a complex dict
        remove_columns=["messages"] 
    )
    
    print(f"Dataset ready with {len(tokenized_dataset)} rows.")
    return tokenized_dataset

In [5]:
#Phase 1 - SFT error on whole i/p
from unsloth import UnslothTrainer, UnslothTrainingArguments
train_dataset = prepare_data_set("/mnt/data/training_data/sales_agent_sft_v2_20260410_003814.jsonl", tokenizer, max_seq_length)
# sft_trainer_p1 = UnslothTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = train_dataset,
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = True,
#     dataset_batched=True,
#     args = UnslothTrainingArguments(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 1,
#         max_steps = 3,
#         learning_rate = 5e-5,
#         logging_steps = 5,
#         # save_steps = 50,
#         # save_total_limit=2,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "cosine",
#         seed = 3407,
#         output_dir = "/mnt/data/outputs/",
#         report_to = "none",
        
#     ),
# )


Map: 100%|██████████| 94/94 [00:00<00:00, 641.20 examples/s]

Dataset ready with 94 rows.


In [6]:
# sft_trainer_p1.train()

In [7]:
from transformers import DataCollatorForLanguageModeling
import torch

class UnifiedCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer, *args, mlm=False, **kwargs)
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        # Filter out 'text' strings to prevent the ValueError
        non_string_examples = []
        for e in examples:
            non_string_examples.append({k: v for k, v in e.items() if k != "text"})
            
        # Let the parent handle the tensor conversion/padding
        batch = super().torch_call(non_string_examples)
        
        # Apply the mask to 'labels'
        for i in range(len(batch["labels"])):
            curr_labels = batch["labels"][i]
            # Find sequence of token IDs
            for idx in range(len(curr_labels) - len(self.response_token_ids)):
                if curr_labels[idx : idx + len(self.response_token_ids)].tolist() == self.response_token_ids:
                    # Mask everything UP TO the end of the assistant header
                    batch["labels"][i][:idx + len(self.response_token_ids)] = -100
                    break
        return batch


# Initialize
collator = UnifiedCompletionCollator(
    response_template="<|im_start|>assistant\n", 
    tokenizer=tokenizer
)

In [20]:
# Test the collator on 1 sample
sample = [train_dataset[0]]
test_batch = collator.torch_call(sample)

# Check how many tokens are NOT masked (-100)
labels = test_batch["labels"][0]
unmasked_count = (labels != -100).sum().item()
total_count = len(labels)

print(f"Total tokens: {total_count}")
print(f"Unmasked tokens (Assistant only): {unmasked_count}")

# Decode the unmasked part to see if it's the <t> block
unmasked_tokens = labels[labels != -100]
print("What the model is learning:")
print(tokenizer.decode(unmasked_tokens))

Total tokens: 611
Unmasked tokens (Assistant only): 30
What the model is learning:
Enterprise Elite is priced at 110$ per month with Unlimited Data. Features include 5G network connectivity, Priority access to high-speed data, Service-level agreement support, and Secure VPN access.<|im_end|>
<|im_end|>


In [9]:
sft_trainer_p2 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,    
    data_collator=collator,
    dataset_text_field = "text",
    # formatting_func = format_sft_custom,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    # dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 8,
        learning_rate = 2e-4,
        logging_steps = 5,
        # save_steps = 15,
        # save_total_limit=3,
        # eval_strategy="steps",  # Tell the trainer to evaluate periodically
        # eval_steps=10,                # Run evaluation every 10 steps
        # metric_for_best_model="loss", # Or "eval_loss" (lower is better)
        # greater_is_better=False,
        # load_best_model_at_end=True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        # completion_only_loss=True,
        # save_strategy="steps"
    ),
)


In [10]:
sft_trainer_p2.train()
# sft_trainer.evaluate()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 94 | Num Epochs = 8 | Total steps = 96
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 475,251,712 of 2,251,611,648 (21.11% trained)


Step,Training Loss
5,6.078965
10,4.272207
15,2.348240
20,1.726647
25,1.031317
30,0.749561
35,0.666741
40,0.438645
45,0.340000
50,0.230801


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=96, training_loss=0.9673134175439676, metrics={'train_runtime': 209.1294, 'train_samples_per_second': 3.596, 'train_steps_per_second': 0.459, 'total_flos': 4909786063349760.0, 'train_loss': 0.9673134175439676, 'epoch': 8.0})

In [11]:
////

SyntaxError: invalid syntax (640674642.py, line 1)

In [19]:
from transformers import TextStreamer
import time
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

SYSTEM_PROMPT_TEMPLATE = """
SALES_AGENT
Working for - amAIz Telecom

## Plan Catalogue

### Core_Essentials
name: Core_Essentials  
desc: Core Essentials  
type: N  

price_int: 30 | price_str: 30$  
data_int: 5 | data_str: 5GB  

features:
- name: 4g  
  desc: 4G network connectivity
- name: basic_voice  
  desc: Standard local voice calling
- name: basic_sms  
  desc: Standard SMS messaging

---

### Urban_Connect
name: Urban_Connect  
desc: Urban Connect  
type: N  

price_int: 60 | price_str: 60$  
data_int: 40 | data_str: 40GB  

features:
- name: 5g  
  desc: 5G network connectivity
- name: unlimited_voice  
  desc: Unlimited local voice calls
- name: hotspot  
  desc: Mobile hotspot support

---

### Apex_Unlimited
name: Apex_Unlimited  
desc: Apex Unlimited  
type: N  

price_int: 90 | price_str: 90$  
data_int: 999 | data_str: Unlimited Data  

features:
- name: 5g  
  desc: 5G network connectivity
- name: priority_data  
  desc: Priority access to high-speed data
- name: international_roaming  
  desc: International roaming

---

### Enterprise_Elite
name: Enterprise_Elite  
desc: Enterprise Elite  
type: B  

price_int: 110 | price_str: 110$  
data_int: 999 | data_str: Unlimited Data  

features:
- name: 5g  
  desc: 5G network connectivity
- name: priority_data  
  desc: Priority access to high-speed data
- name: sla_support  
  desc: Service-level agreement support
- name: vpn  
  desc: Secure VPN access

---

### Student_Basic
name: Student_Basic  
desc: Student Basic  
type: S  

price_int: 25 | price_str: 25$  
data_int: 10 | data_str: 10GB  

features:
- name: 4g  
  desc: 4G network connectivity
- name: social_pass  
  desc: Unlimited access to selected social apps

---

### Senior_Basic
name: Senior_Basic  
desc: Senior Basic  
type: R  

price_int: 35 | price_str: 35$  
data_int: 10 | data_str: 10GB  

features:
- name: 4g  
  desc: 4G network connectivity
- name: priority_care  
  desc: Priority customer support

## CUSTOMER PROFILE
- customer_type: N
- current_plan: Urban_Connect
- average_data_usage: 50GB
THE_END
"""
# Define your Evaluation Questions
QUESTIONS = [
    # 1. Boundary Logic Trap (Target: LOGIC_TRAP)
    # Tests strictly > 60 vs >= 60 logic.
    "show me all your plan names?",
    "explain more about the last one",
    "how about the first one?",
    "i want a plan with unlimited data.",   #Apex Unlimited includes Unlimited Data and is priced at 90$ per month.
    "how much I am paying now?",     # You are currently paying 60$ per month for your current plan, Urban Connect.
    "what all features my plan has", # Urban Connect includes 5G network connectivity, Unlimited local voice calls, and Mobile hotspot support.
    "is there any plan which costs less than 62$ ",#
    "can u compare my plan with the next best option?",
    "am i eligible for a student plan?",  #Student plans are available only to student customers and cannot be applied to your account.
    "who is USA president now?",   #Joe B
    "i dont have money, i want to reduce my bill?",
    "i am starting a business, what plan u can suggest?", #Based on your business needs and average data usage, I would recommend the Apex Unlimited plan. It offers Unlimited Data with 5G network connectivity, Priority access to high-speed data, and International roaming.
    "how much I am paying now? how much for unlimitd data plan?", #Urban Connect is priced at 60$ per month with 40GB data.
    "ewitch from urban comnect to cofe essentials", #Core Essentials is priced at 30$ per month with 5GB data. Based on your usage, this plan may not meet your average data needs.
    "I am starting a business, do u have any plans for business people?" #
]


OUTPUT_FILE = "/mnt/data/training_data/evaluation_results.jsonl"

# --- RUN EVALUATION ---
with open(OUTPUT_FILE, "w") as f:
        questions = []
        responses = []
        for question in QUESTIONS:
            # 1. Prepare Prompt
            
            full_instruction = SYSTEM_PROMPT_TEMPLATE
            messages = [{"role": "system", "content": full_instruction.replace("{{","{").replace("}}","}")}] + \
           [m for q, a in zip(questions, responses) for m in ({"role":"user","content": q}, {"role":"assistant","content": a})] + \
           [{"role":"user","content": question}]

            # 2. Tokenize and Generate
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")

            # Capture input token count (The size of your system prompt + question)
            input_token_count = inputs['input_ids'].shape[1]

            # Start the timer
            start_time = time.time()

            # Execute model generation
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.7,
                eos_token_id=tokenizer.eos_token_id
            )

            # Calculate total response time
            total_response_time = time.time() - start_time
            
            # 3. Decode Response and Capture Output Token Count
            # Slicing the output [len(inputs[0]):] ensures we only count NEW tokens
            generated_tokens = outputs[0][len(inputs[0]):]
            output_token_count = len(generated_tokens)
            
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            responses.append(response_text)
            questions.append(question)
            print(f"Q: {question}\nA: {response_text[:100]}...P:") # Console preview

            # 4. Save to JSONL with Performance Metrics
            eval_entry = {
                
                "question": question,
                "model_response": response_text,
                "metrics": {
                    "input_tokens": int(input_token_count),
                    "output_tokens": int(output_token_count),
                    "total_latency_sec": round(total_response_time, 4),
                    "tokens_per_sec": round(output_token_count / total_response_time, 2) if total_response_time > 0 else 0
                }
            }
            f.write(json.dumps(eval_entry) + "\n")

print(f"\nEvaluation Complete. Results saved to {OUTPUT_FILE}")

Q: show me all your plan names?
A:  I cataloged the available plans as follows:

Core Essentials, Urban Connect, and Apex Unlimited are...P:
Q: explain more about the last one
A:  Apex Unlimited is priced at 90$ per month with Unlimited Data. Features include 5G network connecti...P:
Q: how about the first one?
A:  Core Essentials is priced at 30$ per month with 5GB data. Features include 4G network connectivity,...P:
Q: i want a plan with unlimited data.
A:  Urban Connect includes 40GB data at 60$ per month....P:
Q: how much I am paying now?
A:  You are currently paying 60$ per month for your current plan, Urban Connect....P:
Q: what all features my plan has
A:  Core Essentials includes 4G network connectivity, Standard local voice calling, and Standard SMS me...P:
Q: is there any plan which costs less than 62$ 
A:  Core Essentials is priced at 30$ per month with 5GB data....P:
Q: can u compare my plan with the next best option?
A: ...P:
Q: am i eligible for a student plan?
A:  How ca

In [ ]:
//////

In [ ]:
model_2, tokenizer_2 = FastLanguageModel.from_pretrained(
    model_name = "/mnt/data/outputs/checkpoint-84",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


In [ ]:
model.save_pretrained("/mnt/data/models/amaiz_sales-llm-1.5B_adap")
tokenizer.save_pretrained("/mnt/data/models/amaiz_sales-llm-1.5B_adap")

In [ ]:
# model.save_pretrained_gguf("/mnt/data/models/amaiz_sales-llm-1.5B_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
# for name, param in model.named_parameters():
#     if "lora" in name:
#         if not param.requires_grad:
#             print(f"🚨 WARNING: {name} is FROZEN! Training will not work.")
#         else:
#             print(f"✅ {name} is trainable.")
#         break